In [0]:
%skip
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

for key, value in {
    "smtp_from": "depariksheet2@gmail.com",
    "smtp_to": "depariksheet2@gmail.com",
    "smtp_host": "smtp.gmail.com",
    "smtp_user": "depariksheet2@gmail.com",
    "smtp_password": "jbhkygewkbdxyqdc",
}.items():
    w.secrets.put_secret(scope="alerts", key=key, string_value=value)

In [0]:
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("job_run_id", "")
dbutils.widgets.text("job_name", "")

raw_id = dbutils.widgets.get("job_run_id").strip()
if not raw_id:
    raise ValueError("Enter a job run ID in the 'job_run_id' widget above, then re-run.")

run_id = int(raw_id)
job_name = dbutils.widgets.get("job_name") or "manual-test"

w = WorkspaceClient()
run = w.jobs.get_run(run_id)

failed_tasks = [
    t for t in run.tasks
    if t.state and t.state.result_state
    and t.state.result_state.value in ("FAILED", "TIMEDOUT", "CANCELED")
]

In [0]:
lines = [f"Job: {job_name}", f"Run: {run.run_page_url}", ""]
for t in failed_tasks:
    err = w.jobs.get_run_output(t.run_id).error or t.state.state_message
    lines += [f"Task: {t.task_key}", f"Error: {err}", f"Link: {t.run_page_url}", ""]

lines += ["Regards", "Pariksheet"]

subject = f"[FAILED] {job_name} - run {run_id}"
body = "\n".join(lines)

In [0]:
import smtplib
from email.message import EmailMessage

secret = lambda k: dbutils.secrets.get("alerts", k).strip()

msg = EmailMessage()
msg["Subject"], msg["From"], msg["To"] = subject, secret("smtp_from"), secret("smtp_to")
msg.set_content(body)

with smtplib.SMTP(secret("smtp_host"), 587, timeout=30) as s:
    s.starttls()
    s.login(secret("smtp_user"), secret("smtp_password"))
    s.send_message(msg)